In [1]:
import sys
from pathlib import Path

# Add src directory to system path so we can import local modules
sys.path.append(str(Path.cwd().parent / "src"))

from preprocessing.loader import (
    BharatFluxDataset,
    load_all,
    build_inventory,
)
from preprocessing.cleaner import (
    standardize_column_names,
    standardize_missing_values,
    convert_numeric_types,
    sort_by_doy,
    validate_dataset,
)

In [2]:
DATA_DIR = Path("../data/raw/bharatflux")
datasets = load_all(DATA_DIR)
print(f"Datasets found: {len(datasets)}")

Datasets found: 38


In [3]:
inventory = build_inventory(datasets)
inventory.head(5)

,Dataset,Site,Year,File Type,Rows,Columns,Start DoY,End DoY,Coverage (%),Column Names,Path
0,BFT_2016_LE_ET_dmean,BFT,2016,combined,366,3,1.0,366.0,100.0,"Day, LE_daily_mean (W m-2), ET_daily_mean (mm ...",..\data\raw\bharatflux\BFT\BFT_2016_LE_ET_dmea...
1,BFT_2017_LE_ET_dmean,BFT,2017,combined,365,3,1.0,365.0,100.0,"Day, LE_daily_mean (W m-2), ET_daily_mean (mm ...",..\data\raw\bharatflux\BFT\BFT_2017_LE_ET_dmea...
2,BFT_2018_LE_ET_dmean,BFT,2018,combined,365,3,1.0,365.0,100.0,"Day, LE_daily_mean (W m-2), ET_daily_mean (mm ...",..\data\raw\bharatflux\BFT\BFT_2018_LE_ET_dmea...
3,BIT_2016_LE_ET_dmean,BIT,2016,combined,366,3,1.0,366.0,100.0,"DoY, LE_daily_mean (W m-2), ET_daily_mean (mm ...",..\data\raw\bharatflux\BIT\BIT_2016_LE_ET_dmea...
4,BIT_2017_LE_ET_dmean,BIT,2017,combined,365,3,1.0,365.0,100.0,"DoY, LE_daily_mean (W m-2), ET_daily_mean (mm ...",..\data\raw\bharatflux\BIT\BIT_2017_LE_ET_dmea...


In [4]:
dataset = datasets["BFT_2016_LE_ET_dmean"]
dataset.info

DatasetInfo(site='BFT', year=2016, file_type='combined', path=WindowsPath('../data/raw/bharatflux/BFT/BFT_2016_LE_ET_dmean.csv'))

In [5]:
dataset.info.site

'BFT'

In [6]:
dataset.info.year

2016

In [7]:
dataset.info.file_type

'combined'

In [8]:
dataset.data.head()

,Day,LE_daily_mean (W m-2),ET_daily_mean (mm d-1)
0,1,121.533229,4.666875
1,2,124.119004,4.766169
2,3,104.577071,4.015759
3,4,109.743590,4.214153
4,5,110.055843,4.226144


In [9]:
dataset.data.columns

Index(['Day', 'LE_daily_mean (W m-2)', 'ET_daily_mean (mm d-1)'], dtype='str')

In [10]:
clean_df = standardize_column_names(dataset.data)
clean_df.columns

Index(['DoY', 'LE', 'ET'], dtype='str')

In [11]:
dataset = datasets["BKC_2016_LE_dmean"]
dataset.data.columns

Index(['Day', 'H_daily_mean (W m-2)', 'H_daily_sd (W m-2)'], dtype='str')

In [12]:
clean_df = standardize_column_names(dataset.data)
clean_df.columns

Index(['DoY', 'LE', 'LE_SD'], dtype='str')

In [13]:
dataset = datasets["JIT_2017_ET_LE_dmean"]
df = dataset.data
df = standardize_column_names(df)
df = standardize_missing_values(df)
df = convert_numeric_types(df)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   DoY     365 non-null    int64  
 1   LE      272 non-null    float64
 2   ET      272 non-null    float64
dtypes: float64(2), int64(1)
memory usage: 8.7 KB


In [14]:
df = sort_by_doy(df)
# df.head()
df[df["LE"].notna()].head()

,DoY,LE,ET
93,94,-0.056394,-0.002144
94,95,-2.231762,-0.084843
95,96,-2.011335,-0.076463
96,97,-2.878423,-0.109426
97,98,1.438659,0.054692


In [15]:
dataset = datasets["JIT_2017_ET_LE_dmean"]

df = dataset.data

df = standardize_column_names(df)
df = standardize_missing_values(df)
df = convert_numeric_types(df)
df = sort_by_doy(df)

cleaned_dataset = BharatFluxDataset(
    info=dataset.info,
    data=df,
)

report = validate_dataset(cleaned_dataset)

report

ValidationReport(site='JIT', year=2017, rows=365, start_doy=1, end_doy=365, duplicate_doy=0, missing_le=93, missing_et=93, doy_coverage_percent=100.0, sorted=True, valid=True)